In [ ]:
!pip install mediapipe==0.10.14

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 24.6 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: mediapipe
    Found existing installation: mediapipe 0.10.32
    Uninstalling mediapipe-0.10.32:
      Successfully uninstalled mediapipe-0.10.32
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 4.25.8 which is incompatible.
ydf 0.14.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.8 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.8 which is incompatible.
grpcio-status 1.71.2 requires proto

In [ ]:
# ==========================================
# NOTEBOOK 1: SKELETON EXTRACTION & NORMALIZATION
# ==========================================
import os
import cv2
import mediapipe as mp
import csv
import numpy as np
from google.colab import drive



# CHANGE THIS: Set to 1 for Account A, 2 for Account B
PART = 1

DATA_DIR = '/content/drive/MyDrive/SignApp/asl_alphabet' # Update path
OUTPUT_FILE = f'/content/drive/MyDrive/SignApp/landmarks_part{PART}.csv'

# --- 2. MEDIAPIPE SETUP ---
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.5)

# --- 3. HELPER: NORMALIZATION ---
def normalize_landmarks(landmarks):
    # Convert to numpy (21 x 3)
    coords = np.array([[lm.x, lm.y, lm.z] for lm in landmarks.landmark])

    # Root Centering (Wrist at 0,0,0)
    wrist = coords[0]
    coords = coords - wrist

    # Scale Normalization (Wrist to Middle Finger MCP = 1.0)
    scale_dist = np.linalg.norm(coords[0] - coords[9])
    if scale_dist < 1e-6: scale_dist = 1
    coords = coords / scale_dist

    return coords.flatten()

# --- 4. PREPARE CSV ---
header = ['label']
for i in range(21):
    header.extend([f'x{i}', f'y{i}', f'z{i}'])

with open(OUTPUT_FILE, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(header)

    # --- 5. SPLIT STRATEGY ---
    all_classes = sorted(os.listdir(DATA_DIR))
    mid_point = len(all_classes) // 2

    if PART == 1:
        my_classes = all_classes[:mid_point]  # First Half
    else:
        my_classes = all_classes[mid_point:]  # Second Half

    print(f"✅ Processing Part {PART}: {my_classes[0]} to {my_classes[-1]}")

    # --- 6. PROCESSING LOOP ---
    for label in my_classes:
        class_path = os.path.join(DATA_DIR, label)
        if not os.path.isdir(class_path): continue

        print(f"Processing: {label}")

        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)
            img = cv2.imread(img_path)
            if img is None: continue

            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            results = hands.process(img_rgb)

            if results.multi_hand_landmarks and results.multi_handedness:
                for hand_lms, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):

                    # Mirror Trick: If "Left", flip X to make it "Right"
                    if handedness.classification[0].label == 'Left':
                        for lm in hand_lms.landmark:
                            lm.x = 1.0 - lm.x

                    # Normalize & Save
                    norm_features = normalize_landmarks(hand_lms)
                    row = [label] + norm_features.tolist()
                    writer.writerow(row)

print(f"✅ Part {PART} Complete! Saved to {OUTPUT_FILE}")

✅ Processing Part 1: A to N
Processing: A


/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Processing: B
Processing: C
Processing: D
Processing: E
Processing: F
Processing: G
Processing: H
Processing: I
Processing: J
Processing: K
Processing: L
Processing: M
Processing: N
✅ Part 1 Complete! Saved to /content/drive/MyDrive/SignApp/landmarks_part1.csv


In [ ]:
import pandas as pd
import os


# 2. DEFINE FILE PATHS
# Make sure these match exactly where you saved Part 1 and Part 2
base_path = '/content/drive/MyDrive/SignApp/'
part1_path = os.path.join(base_path, 'landmarks_part1.csv')
part2_path = os.path.join(base_path, 'landmarks_part2.csv')
output_path = os.path.join(base_path, 'landmarks_normalized.csv')

# 3. CHECK IF FILES EXIST
if not os.path.exists(part1_path) or not os.path.exists(part2_path):
    print("❌ Error: Could not find one or both part files.")
    print(f"Looking for:\n  {part1_path}\n  {part2_path}")
else:
    # 4. LOAD AND MERGE
    print("Loading Part 1...")
    df1 = pd.read_csv(part1_path)

    print("Loading Part 2...")
    df2 = pd.read_csv(part2_path)

    print("Merging...")
    full_df = pd.concat([df1, df2], axis=0, ignore_index=True)

    # 5. SAVE THE FINAL FILE
    full_df.to_csv(output_path, index=False)

    # 6. VERIFICATION
    print("-" * 30)
    print(f"✅ SUCCESS! File saved to: {output_path}")
    print(f"Total Images: {len(full_df)}")
    print(f"Unique Classes: {full_df['label'].nunique()} (Should be 29)")
    print("-" * 30)
    print("You can now run Notebook 2 normally.")

Loading Part 1...
Loading Part 2...
Merging...
------------------------------
✅ SUCCESS! File saved to: /content/drive/MyDrive/SignApp/landmarks_normalized.csv
Total Images: 63677
Unique Classes: 29 (Should be 29)
------------------------------
You can now run Notebook 2 normally.
